# Analyse Exlcuded Feature From Sub Trees

Take parameters:
- Folder name (specify which model to analyse)
- Count (Number of trees to analyse starting from index 0)
- Produces excluded 

In [ ]:
import pydot
import pandas as pd
from collections import deque


In [ ]:
def getRealNodes(nodes):
    real_nodes = []
    for node in nodes:
        if node.get_attributes().get("label") is not None:
            real_nodes.append(node)
    return real_nodes

In [ ]:
def jsonify_nodes(root_node_name, node_list, edge_list):
    """
    Traverses a pydot graph from a root node and returns a list of dictionaries 
    with each node's label and level.

    Args:
        root_node_name (str): The name of the starting (root) node.
        node_list (list): A list of pydot Node objects.
        edge_list (list): A list of pydot Edge objects.

    Returns:
        list: A list of dictionaries, each containing a node's 'label' and 'level'.
    """
    # 1. Build an adjacency list for efficient neighbor lookup
    adj = {node.get_name(): [] for node in node_list}
    for edge in edge_list:
        source = edge.get_source()
        destination = edge.get_destination()
        if source in adj:
            adj[source].append(destination)
    
    # 2. Map node names to their labels
    # Use .get_label() but handle the case where it returns a name, which occurs
    # if no explicit label is set.
    name_to_label_map = {
        node.get_name(): node.get_attributes().get("label")
        if node.get_label() != node.get_name() else node.get_name()
        for node in node_list
    }

    # 3. Prepare data structures for BFS
    queue = deque([(root_node_name, 0)])  # Store tuples of (node_name, level)
    visited = {root_node_name}
    result_list = []

    # 4. Perform the BFS traversal
    while queue:
        current_node_name, level = queue.popleft()
        
        # Get the label from the map we created
        current_label = name_to_label_map.get(current_node_name, current_node_name)
        
        # Add the current node's label and level to the result
        result_list.append({
            "label": current_label,
            "level": level
        })

        # Enqueue all unvisited neighbors
        for neighbor_name in adj.get(current_node_name, []):
            if neighbor_name not in visited:
                visited.add(neighbor_name)
                queue.append((neighbor_name, level + 1))
    
    return result_list

In [ ]:
def tabulate_nodes(json_nodes):
    for node in json_nodes:
        node['label'] = node['label'].split(" ")[0].replace('"', '').replace('<', '')
        
    return pd.DataFrame(json_nodes)

test_list = [{
    "label": "BW <= ashdjkasdhk"
}]

tabulate_nodes(test_list)

In [ ]:
def analyse_excluded(folderName: str, count: int):
    df = pd.read_csv('../data/' + 'sample.csv', low_memory=False)
    
    for i in range(count):
        graph = pydot.graph_from_dot_file(f'../../application/public/dot/{folderName}/tree_{i:03d}.dot')
        real_nodes = getRealNodes(graph[0].get_nodes())
        jsonified_nodes = jsonify_nodes(real_nodes[0].get_name(), graph[0].get_nodes(), graph[0].get_edges())
        tabulated_nodes = tabulate_nodes(jsonified_nodes)
        # display(tabulated_nodes)
        feature_set = tabulated_nodes["label"].to_list()
        
        for idx, feature in enumerate(feature_set):
            if feature == "AtSteroid":
                feature_set[idx] = "AtSteroid Dose_term"
            if feature == "Intrapartum":
                feature_set[idx] = "Intrapartum Antibiotic_term"
        
        diff = set(df.columns.to_list()) - set(feature_set)
        print(f"Tree {i:03d} excluded {len(diff)} features: {diff}")

In [ ]:
analyse_excluded('extra_trees', 2)

In [ ]:
analyse_excluded('gradient_boosting', 2)

In [ ]:
analyse_excluded('random_forest', 2)